# VisualSurface — Trading Desk Readiness Evaluation

This notebook loads a model checkpoint and runs a comprehensive evaluation suite to determine whether the IV surface reconstructor is ready for production use on a trading desk.

**Sections:**
1. Setup & Checkpoint Loading
2. Training Curve Review
3. Surface Quality — Single-Day Deep Dive
4. Quote Fitting Metrics (RMSE, MAE, R², bias)
5. Residual Analysis (by moneyness, maturity, IV level)
6. Bid-Ask Calibration (% quotes fit within spread)
7. No-Arbitrage Analysis (violations breakdown)
8. Smoothness & Curvature
9. Smile Shape Metrics (ATM vol, skew, kurtosis per expiry)
10. Term Structure Analysis
11. Greeks Consistency
12. Cross-Sectional Stability (vol-of-vol across days)
13. Trading Desk Readiness Scorecard

## 0 — Configuration

In [ ]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "checkpoints/last.ckpt"        # path to checkpoint to evaluate
DATA_PATH       = "./data/108105/2025_C_options_data.csv"
KFWD_MIN        = 0.15
KFWD_MAX        = 1.50
DEVICE          = "cpu"                           # "mps" if on Apple Silicon
MAX_VAL_DAYS    = None                            # None = all; set e.g. 50 for speed
SEED            = 42
# ──────────────────────────────────────────────────────────────────────────────

## 1 — Imports & Setup

In [ ]:
from __future__ import annotations

import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats as scipy_stats

import lightning as l
l.seed_everything(SEED, workers=True)

from visualsurface import LitSurfaceModel, SurfaceDataModule
from visualsurface.math_ops import (
    bs_call_from_fwd,
    make_uv_grid,
    no_arb_penalty_from_call_prices,
    sample_iv_grid_at_quotes,
    v_to_t_years,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
print("Imports OK")

## 2 — Load Data Module & Checkpoint

In [ ]:
dm = SurfaceDataModule(
    data_path=DATA_PATH,
    num_workers=0,
    kfwd_min=KFWD_MIN,
    kfwd_max=KFWD_MAX,
)
dm.setup("fit")
spec = dm.spec

print(f"RasterSpec : Nu={spec.Nu}, Nv={spec.Nv}")
print(f"  u ∈ [{spec.u_min:.3f}, {spec.u_max:.3f}]  (log-moneyness)")
print(f"  v ∈ [{spec.v_min:.3f}, {spec.v_max:.3f}]  (log-time)")
print(f"Train days : {len(dm.train_ds)}")
print(f"Val days   : {len(dm.val_ds)}")

In [ ]:
lit = LitSurfaceModel.load_from_checkpoint(
    CHECKPOINT_PATH,
    spec=spec,
    map_location=DEVICE,
)
lit.eval()
lit.to(DEVICE)

total_params = sum(p.numel() for p in lit.parameters())
trainable    = sum(p.numel() for p in lit.parameters() if p.requires_grad)
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Parameters : {total_params:,}  ({trainable:,} trainable)")
print(f"Hparams    : {dict(lit.hparams)}")

## 3 — Training Curve Review

In [ ]:
import glob, os, csv

def load_tensorboard_csv(log_dir: str) -> pd.DataFrame | None:
    """Load metrics from the latest Lightning CSV logger version."""
    versions = sorted(glob.glob(os.path.join(log_dir, "version_*")))
    if not versions:
        return None
    latest = versions[-1]
    metrics_file = os.path.join(latest, "metrics.csv")
    if not os.path.exists(metrics_file):
        return None
    df = pd.read_csv(metrics_file)
    print(f"Loaded metrics from: {metrics_file}  ({len(df)} rows)")
    return df

metrics_df = load_tensorboard_csv("lightning_logs")

if metrics_df is not None:
    train_cols = [c for c in metrics_df.columns if c.startswith("train/")]
    val_cols   = [c for c in metrics_df.columns if c.startswith("val/")]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    components = [("fit", "Fitting Loss (MSE @ quotes)"),
                  ("smooth", "Smoothness Loss"),
                  ("arb", "No-Arb Penalty")]

    for ax, (key, title) in zip(axes, components):
        tr_col = f"train/{key}"
        vl_col = f"val/{key}"
        if tr_col in metrics_df.columns:
            t = metrics_df[["epoch", tr_col]].dropna()
            ax.plot(t["epoch"], t[tr_col], label="Train", color="steelblue")
        if vl_col in metrics_df.columns:
            v = metrics_df[["epoch", vl_col]].dropna()
            ax.plot(v["epoch"], v[vl_col], label="Val", color="tomato")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()

    fig.suptitle("Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No metrics CSV found — skipping training curve plot.")

## 4 — Full Validation Pass  
Run the model on every validation day and collect per-quote residuals.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class DayResult:
    date: str
    rmse: float
    mae: float
    bias: float
    r2: float
    pct_within_spread: float
    n_quotes: int
    arb_mono: float
    arb_convex: float
    arb_cal: float
    curvature: float
    # per-quote arrays
    residuals: np.ndarray = field(repr=False, default_factory=lambda: np.array([]))
    obs_iv: np.ndarray    = field(repr=False, default_factory=lambda: np.array([]))
    pred_iv: np.ndarray   = field(repr=False, default_factory=lambda: np.array([]))
    quote_u: np.ndarray   = field(repr=False, default_factory=lambda: np.array([]))
    quote_v: np.ndarray   = field(repr=False, default_factory=lambda: np.array([]))
    bid: np.ndarray       = field(repr=False, default_factory=lambda: np.array([]))
    ask: np.ndarray       = field(repr=False, default_factory=lambda: np.array([]))
    spread: np.ndarray    = field(repr=False, default_factory=lambda: np.array([]))
    iv_grid: np.ndarray   = field(repr=False, default_factory=lambda: np.array([]))


def evaluate_day(row: dict, dm: SurfaceDataModule, lit: LitSurfaceModel) -> DayResult:
    batch = dm.collate_fn([row]).to(DEVICE)
    with torch.no_grad():
        iv_grid = lit.model(
            batch.img, batch.quote_u, batch.quote_v,
            batch.quote_num, batch.cp, batch.style,
            batch.quote_valid, batch.global_feats,
        )[0]  # [Nv, Nu]

    valid = batch.quote_valid[0]
    pred_at_q = sample_iv_grid_at_quotes(
        iv_grid.unsqueeze(0),
        batch.quote_u,
        batch.quote_v,
        spec,
    )[0][valid].cpu().numpy()

    obs   = batch.quote_iv[0][valid].cpu().numpy()
    qu    = batch.quote_u[0][valid].cpu().numpy()
    qv    = batch.quote_v[0][valid].cpu().numpy()

    # raw bid/ask not stored in batch; recover from quote_num (unnormalized isn't stored)
    # We approximate bid-ask using the quote_num feature 2 (spread/spot) and spot
    spot  = float(batch.spot[0])
    # quote_num channels (raw, before normalization): [iv,mid/S,spread/S,price/S,delta,gamma*S2,r,q,logK/S,vix]
    # We stored raw Bid/Ask in row
    n_valid = int(valid.sum())
    bid_raw = np.array(row["Bid_list"][:n_valid], dtype=np.float32)
    ask_raw = np.array(row["Ask_list"][:n_valid], dtype=np.float32)
    spread  = np.clip(ask_raw - bid_raw, 1e-8, None)

    res   = pred_at_q - obs
    rmse  = float(np.sqrt((res**2).mean()))
    mae   = float(np.abs(res).mean())
    bias  = float(res.mean())
    ss_res = ((res)**2).sum()
    ss_tot = ((obs - obs.mean())**2).sum()
    r2    = 1.0 - ss_res / (ss_tot + 1e-12)

    # bid-ask calibration: is the IV residual within the option price spread converted to vol?
    # Approximate: spread in IV ≈ spread_price / (vega); we compare directly in price space
    # Simple: count residuals whose absolute value < 0.5 * spread/spot (rough IV equiv)
    # Better: just count |pred_iv - obs_iv| < half_spread_iv where we use the raw IV spread
    # We use: quote within spread if |pred - obs| < (ask_iv - bid_iv) / 2
    # IV spread proxy: we only have price spread; skip for now and use price check
    # Use abs(pred - obs) < spread/spot as a rough check
    spread_iv_proxy = spread / max(spot, 1e-3)
    pct_within = float(np.mean(np.abs(res) <= 0.5 * spread_iv_proxy))

    # No-arb decomposition
    u_vec, v_vec = make_uv_grid(spec, device=iv_grid.device)
    T_vec = torch.clamp(v_to_t_years(v_vec), min=1/365)
    r_med = float(batch.r_q[0][valid].mean()) if valid.sum() > 0 else 0.0
    q_med = float(batch.q_q[0][valid].mean()) if valid.sum() > 0 else 0.0
    T  = T_vec.view(1, spec.Nv, 1)
    S  = batch.spot[0].view(1, 1, 1).to(iv_grid.device)
    Fwd   = S * math.exp((r_med - q_med)) * torch.exp((r_med - q_med) * (T - 1))
    Fwd   = S * torch.exp(torch.tensor(r_med - q_med) * T)
    Disc  = torch.exp(-torch.tensor(r_med) * T)
    K_grid = Fwd * torch.exp(u_vec.view(1, 1, spec.Nu).to(iv_grid.device))
    call   = bs_call_from_fwd(Fwd, K_grid, iv_grid.unsqueeze(0), T, Disc)

    import torch.nn.functional as F_nn
    mono    = F_nn.relu(call[:, :, 1:] - call[:, :, :-1])
    mono_v  = float((mono**2).mean())
    bf      = call[:, :, :-2] - 2*call[:, :, 1:-1] + call[:, :, 2:]
    conv_v  = float((F_nn.relu(-bf)**2).mean())
    cal     = call[:, 1:, :] - call[:, :-1, :]
    cal_v   = float((F_nn.relu(-cal)**2).mean())

    # Curvature (smoothness)
    iv_np = iv_grid.cpu().numpy()
    d2u = iv_np[:, 2:] - 2*iv_np[:, 1:-1] + iv_np[:, :-2]
    d2v = iv_np[2:, :] - 2*iv_np[1:-1, :] + iv_np[:-2, :]
    curv = float((d2u**2).mean() + (d2v**2).mean())

    date = str(row.get("date", "?"))
    return DayResult(
        date=date, rmse=rmse, mae=mae, bias=bias, r2=r2,
        pct_within_spread=pct_within, n_quotes=n_valid,
        arb_mono=mono_v, arb_convex=conv_v, arb_cal=cal_v,
        curvature=curv,
        residuals=res, obs_iv=obs, pred_iv=pred_at_q,
        quote_u=qu, quote_v=qv,
        bid=bid_raw, ask=ask_raw, spread=spread,
        iv_grid=iv_np,
    )


print("Evaluating validation set...")
val_rows = dm.val_ds.rows
if MAX_VAL_DAYS is not None:
    val_rows = val_rows[:MAX_VAL_DAYS]

results: List[DayResult] = []
for i, row in enumerate(val_rows):
    r = evaluate_day(row, dm, lit)
    results.append(r)
    if (i+1) % 10 == 0 or (i+1) == len(val_rows):
        print(f"  {i+1}/{len(val_rows)}  median RMSE so far: {np.median([x.rmse for x in results]):.4f}")

print(f"Done. Evaluated {len(results)} validation days.")

## 5 — Summary Statistics

In [ ]:
summary = pd.DataFrame([{
    "date": r.date,
    "n_quotes": r.n_quotes,
    "rmse": r.rmse,
    "mae": r.mae,
    "bias": r.bias,
    "r2": r.r2,
    "pct_within_spread": r.pct_within_spread,
    "arb_mono": r.arb_mono,
    "arb_convex": r.arb_convex,
    "arb_cal": r.arb_cal,
    "curvature": r.curvature,
} for r in results])

desc = summary.drop(columns=["date"]).describe(percentiles=[.05, .25, .5, .75, .95])
print(desc.to_string())

## 6 — Core Fitting Metric Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

def hist_with_stats(ax, data, title, xlabel, color="steelblue", bins=40):
    ax.hist(data, bins=bins, color=color, edgecolor="white", linewidth=0.3)
    ax.axvline(np.median(data), color="red", linestyle="--", linewidth=1.5, label=f"Median={np.median(data):.4f}")
    ax.axvline(np.mean(data),   color="orange", linestyle=":",  linewidth=1.5, label=f"Mean={np.mean(data):.4f}")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=9)

hist_with_stats(axes[0,0], summary["rmse"],  "RMSE per Day",             "RMSE (IV units)",   "steelblue")
hist_with_stats(axes[0,1], summary["mae"],   "MAE per Day",              "MAE (IV units)",    "teal")
hist_with_stats(axes[0,2], summary["bias"],  "Bias per Day (pred−obs)",  "Bias (IV units)",   "darkorchid")
hist_with_stats(axes[1,0], summary["r2"],    "R² per Day",               "R²",                "goldenrod")
hist_with_stats(axes[1,1], summary["pct_within_spread"], "% Quotes Within Bid-Ask", "%", "seagreen")
hist_with_stats(axes[1,2], summary["n_quotes"], "Quote Count per Day",   "# quotes",          "slategray")

fig.suptitle("Per-Day Fitting Metric Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7 — Residual Analysis: All Quotes

In [ ]:
all_res  = np.concatenate([r.residuals for r in results])
all_obs  = np.concatenate([r.obs_iv    for r in results])
all_pred = np.concatenate([r.pred_iv   for r in results])
all_qu   = np.concatenate([r.quote_u   for r in results])
all_qv   = np.concatenate([r.quote_v   for r in results])
all_T    = np.exp(all_qv) * 365.0   # days
all_kfwd = np.exp(all_qu)           # K/Fwd

print(f"Total quotes evaluated : {len(all_res):,}")
print(f"Overall RMSE           : {np.sqrt((all_res**2).mean()):.5f}")
print(f"Overall MAE            : {np.abs(all_res).mean():.5f}")
print(f"Overall Bias           : {all_res.mean():.5f}")
ss_res = ((all_res)**2).sum()
ss_tot = ((all_obs - all_obs.mean())**2).sum()
print(f"Overall R²             : {1 - ss_res/ss_tot:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pred vs actual scatter
ax = axes[0]
idx = np.random.choice(len(all_obs), min(8000, len(all_obs)), replace=False)
ax.scatter(all_obs[idx], all_pred[idx], s=3, alpha=0.3, c="steelblue")
lo, hi = all_obs.min(), all_obs.max()
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Perfect fit")
ax.set_xlabel("Observed IV")
ax.set_ylabel("Predicted IV")
ax.set_title("Predicted vs Observed IV")
ax.legend()

# Residual histogram
ax = axes[1]
ax.hist(all_res, bins=80, color="steelblue", edgecolor="white", linewidth=0.2, density=True)
x = np.linspace(all_res.min(), all_res.max(), 300)
mu, sigma = all_res.mean(), all_res.std()
ax.plot(x, scipy_stats.norm.pdf(x, mu, sigma), "r-", linewidth=2, label=f"N(μ={mu:.4f}, σ={sigma:.4f})")
ax.set_xlabel("Residual (pred − obs) IV")
ax.set_ylabel("Density")
ax.set_title("Residual Distribution")
ax.legend()

# Q-Q plot
ax = axes[2]
quantiles = np.percentile(all_res, np.linspace(1, 99, 200))
theoretical = scipy_stats.norm.ppf(np.linspace(0.01, 0.99, 200)) * sigma + mu
ax.scatter(theoretical, quantiles, s=8, alpha=0.6, color="steelblue")
ax.plot([theoretical.min(), theoretical.max()],
        [theoretical.min(), theoretical.max()], "r--", linewidth=1.5)
ax.set_xlabel("Theoretical Normal Quantiles")
ax.set_ylabel("Sample Quantiles")
ax.set_title("Q-Q Plot (Residuals)")

fig.suptitle("Overall Residual Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8 — Residuals by Moneyness & Maturity

In [ ]:
# ── Moneyness buckets ──────────────────────────────────────────────────────────
money_bins   = [0.70, 0.85, 0.95, 1.00, 1.05, 1.15, 1.30, 2.00]
money_labels = ["<0.85", "0.85-0.95", "0.95-1.00", "ATM", "1.00-1.05", "1.05-1.15", "1.15-1.30", ">1.30"]
money_bins   = [0.0, 0.85, 0.95, 1.00, 1.05, 1.15, 1.30, 10.0]
money_labels = ["<0.85", "0.85-0.95", "0.95-1.00", "ATM±0", "1.00-1.05", "1.05-1.15", "1.15-1.30", ">1.30"]

# ── Maturity buckets ──────────────────────────────────────────────────────────
mat_bins   = [0, 14, 30, 60, 90, 180, 365, 1e9]
mat_labels = ["≤14d", "14-30d", "30-60d", "60-90d", "90-180d", "180-365d", ">365d"]

money_idx = np.digitize(all_kfwd, money_bins) - 1
mat_idx   = np.digitize(all_T,    mat_bins)   - 1

def bucket_stats(idx_arr, labels, max_idx=None):
    rows = []
    n_labels = len(labels)
    for i, label in enumerate(labels):
        mask = idx_arr == i
        if mask.sum() == 0:
            continue
        res_b = all_res[mask]
        rows.append({
            "bucket": label,
            "n": int(mask.sum()),
            "rmse": float(np.sqrt((res_b**2).mean())),
            "mae":  float(np.abs(res_b).mean()),
            "bias": float(res_b.mean()),
            "p95":  float(np.percentile(np.abs(res_b), 95)),
        })
    return pd.DataFrame(rows)

money_df = bucket_stats(money_idx, money_labels)
mat_df   = bucket_stats(mat_idx,   mat_labels)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

def plot_bucket_bars(ax, df, col, title, ylabel, color="steelblue"):
    bars = ax.bar(df["bucket"], df[col], color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(df["bucket"], rotation=30, ha="right")
    for bar, val in zip(bars, df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

plot_bucket_bars(axes[0,0], money_df, "rmse", "RMSE by Moneyness (K/Fwd)", "RMSE", "steelblue")
plot_bucket_bars(axes[0,1], money_df, "bias", "Bias by Moneyness (K/Fwd)", "Bias (pred-obs)", "darkorchid")
plot_bucket_bars(axes[1,0], mat_df,   "rmse", "RMSE by Maturity",          "RMSE", "teal")
plot_bucket_bars(axes[1,1], mat_df,   "bias", "Bias by Maturity",          "Bias (pred-obs)", "goldenrod")

fig.suptitle("Residuals by Moneyness & Maturity", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nMoneyness Buckets:")
print(money_df.to_string(index=False, float_format="{:.5f}".format))
print("\nMaturity Buckets:")
print(mat_df.to_string(index=False, float_format="{:.5f}".format))

## 9 — 2D Residual Heatmap (Moneyness × Maturity)

In [ ]:
# Build a 2D grid of mean absolute error
grid_rmse = np.full((len(mat_labels), len(money_labels)), np.nan)
grid_bias = np.full((len(mat_labels), len(money_labels)), np.nan)
grid_n    = np.zeros((len(mat_labels), len(money_labels)))

for mi in range(len(money_labels)):
    for ti in range(len(mat_labels)):
        mask = (money_idx == mi) & (mat_idx == ti)
        if mask.sum() < 5:
            continue
        res_b = all_res[mask]
        grid_rmse[ti, mi] = np.sqrt((res_b**2).mean())
        grid_bias[ti, mi] = res_b.mean()
        grid_n[ti, mi]    = mask.sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, data, title, cmap in [
    (axes[0], grid_rmse, "RMSE Heatmap (Maturity × Moneyness)", "YlOrRd"),
    (axes[1], grid_bias, "Bias Heatmap (pred-obs)",             "RdBu_r"),
]:
    vmax = np.nanmax(np.abs(data)) if np.any(~np.isnan(data)) else 1.0
    if "Bias" in title:
        norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
        im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    else:
        im = ax.imshow(data, cmap=cmap, aspect="auto")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_xticks(range(len(money_labels)))
    ax.set_xticklabels(money_labels, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(mat_labels)))
    ax.set_yticklabels(mat_labels, fontsize=9)
    ax.set_xlabel("Moneyness K/Fwd")
    ax.set_ylabel("Maturity")
    ax.set_title(title)

fig.suptitle("2D Error Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 10 — Bid-Ask Calibration

In [ ]:
all_spread = np.concatenate([r.spread for r in results])
all_bid    = np.concatenate([r.bid    for r in results])
all_ask    = np.concatenate([r.ask    for r in results])

# For each quote: check if predicted price (from predicted IV) fits within bid-ask
# We use the IV residual vs the mid-point and compare to half-spread in IV units
# Rough conversion: Δprice ≈ vega * ΔIV  but we use the observed ratio spread/mid
mid_price = 0.5 * (all_bid + all_ask)
rel_spread = np.where(mid_price > 1e-3, all_spread / mid_price, np.nan)

# Define bid-ask fit in IV space as |residual| < 0.5 * rel_spread * obs_iv
iv_half_spread = 0.5 * rel_spread * all_obs
within = np.abs(all_res) <= iv_half_spread

print(f"% quotes within bid-ask (IV-equiv): {100*within.mean():.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: absolute residual vs half-spread
ax = axes[0]
idx = np.random.choice(len(all_res), min(5000, len(all_res)), replace=False)
clr = np.where(within[idx], "steelblue", "tomato")
ax.scatter(iv_half_spread[idx], np.abs(all_res[idx]), s=4, c=clr, alpha=0.4)
lim = max(np.nanpercentile(iv_half_spread, 99), np.percentile(np.abs(all_res), 99))
ax.plot([0, lim], [0, lim], "k--", linewidth=1.5, label="Boundary")
ax.set_xlabel("Half-Spread (IV equiv)")
ax.set_ylabel("|Residual|")
ax.set_title(f"Bid-Ask Calibration\n(blue=within, red=outside)\n{100*within.mean():.1f}% within")
ax.legend()
ax.set_xlim(0, lim); ax.set_ylim(0, lim)

# By moneyness
ax = axes[1]
bucket_pct = []
for i, label in enumerate(money_labels):
    mask = money_idx == i
    if mask.sum() < 5:
        bucket_pct.append((label, np.nan))
        continue
    bucket_pct.append((label, 100*within[mask].mean()))
labels_b, pcts = zip(*bucket_pct)
colors = ["tomato" if (p is not None and not np.isnan(p) and p < 50) else "steelblue" for p in pcts]
bars = ax.bar(labels_b, pcts, color=colors, edgecolor="white")
ax.axhline(50, color="red", linestyle="--", linewidth=1)
ax.set_xticklabels(labels_b, rotation=30, ha="right")
ax.set_ylabel("% within bid-ask")
ax.set_title("Bid-Ask Fit Rate by Moneyness")
ax.set_ylim(0, 105)

fig.suptitle("Bid-Ask Calibration Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 11 — No-Arbitrage Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

arb_names = [("arb_mono",   "Monotonicity Violation",  "steelblue"),
             ("arb_convex", "Butterfly (Convexity)",   "teal"),
             ("arb_cal",    "Calendar Spread",          "goldenrod")]

for ax, (col, title, color) in zip(axes, arb_names):
    vals = summary[col].values
    ax.hist(vals, bins=40, color=color, edgecolor="white", linewidth=0.3)
    ax.axvline(np.median(vals), color="red", linestyle="--", linewidth=1.5,
               label=f"Median={np.median(vals):.2e}")
    ax.set_title(title)
    ax.set_xlabel("Penalty Value")
    ax.legend(fontsize=9)

fig.suptitle("No-Arbitrage Penalty Distributions (per day)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Per-day time series
fig, ax = plt.subplots(figsize=(14, 4))
dates = pd.to_datetime(summary["date"])
ax.plot(dates, summary["arb_mono"],   label="Monotonicity", alpha=0.8)
ax.plot(dates, summary["arb_convex"], label="Convexity",    alpha=0.8)
ax.plot(dates, summary["arb_cal"],    label="Calendar",     alpha=0.8)
ax.set_xlabel("Date")
ax.set_ylabel("Penalty")
ax.set_title("No-Arb Penalties Over Time")
ax.legend()
plt.tight_layout()
plt.show()

## 12 — Smile Shape Metrics  
For each validation day, extract ATM vol, 25Δ risk-reversal, 25Δ butterfly per expiry.

In [ ]:
u_vec, v_vec = make_uv_grid(spec)
u_arr = u_vec.numpy()    # log(K/Fwd)
v_arr = v_vec.numpy()
T_arr = np.exp(v_arr) * 365.0  # days

atm_idx = int(np.argmin(np.abs(u_arr)))   # closest grid point to log(K/Fwd)=0

# For a simple proxy: 25D put ~ log(K/Fwd) ≈ -0.12 for 30d; we use fixed offsets
# Use moneyness offsets ±0.1 log-units as proxies for wings
wing_offset = 0.10
lo_idx  = int(np.argmin(np.abs(u_arr - (-wing_offset))))
hi_idx  = int(np.argmin(np.abs(u_arr -  wing_offset)))

smile_rows = []
for r in results:
    iv_g = r.iv_grid  # [Nv, Nu]
    for ti, T_days in enumerate(T_arr):
        atm_vol = float(iv_g[ti, atm_idx])
        lo_vol  = float(iv_g[ti, lo_idx])
        hi_vol  = float(iv_g[ti, hi_idx])
        rr      = lo_vol - hi_vol          # risk-reversal (put wing - call wing)
        fly     = 0.5*(lo_vol + hi_vol) - atm_vol  # butterfly
        smile_rows.append({"date": r.date, "T_days": float(T_days),
                           "atm_vol": atm_vol, "rr": rr, "fly": fly})

smile_df = pd.DataFrame(smile_rows)

# Plot by maturity bucket
exp_buckets = [(7, 21), (21, 45), (45, 90), (90, 180), (180, 365)]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for label, col, ax in [("ATM Vol", "atm_vol", axes[0]),
                        ("Risk-Reversal (±0.1 log)", "rr", axes[1]),
                        ("Butterfly (±0.1 log)",    "fly", axes[2])]:
    for lo_t, hi_t in exp_buckets:
        sub = smile_df[(smile_df["T_days"] >= lo_t) & (smile_df["T_days"] < hi_t)]
        if sub.empty:
            continue
        vals = sub[col].dropna()
        ax.hist(vals, bins=30, alpha=0.5, label=f"{lo_t}-{hi_t}d", density=True)
    ax.set_title(label)
    ax.set_xlabel("IV")
    ax.legend(fontsize=8)

fig.suptitle("Smile Shape Statistics by Expiry Bucket", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 13 — Single-Day Deep Dive (Best / Median / Worst)

In [ ]:
def visualize_single_day(result: DayResult, title_prefix: str = ""):
    iv_g  = result.iv_grid   # [Nv, Nu]
    qu    = result.quote_u
    qv    = result.quote_v
    obs   = result.obs_iv
    res   = result.residuals
    kfwd  = np.exp(qu)
    T_days_q = np.exp(qv) * 365.0

    u_arr_g = np.linspace(spec.u_min, spec.u_max, spec.Nu)
    v_arr_g = np.linspace(spec.v_min, spec.v_max, spec.Nv)
    kfwd_g  = np.exp(u_arr_g)
    T_days_g = np.exp(v_arr_g) * 365.0
    KK, TT = np.meshgrid(kfwd_g, T_days_g)

    fig = plt.figure(figsize=(18, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

    # 1. 2D heatmap
    ax0 = fig.add_subplot(gs[0, 0])
    im = ax0.imshow(iv_g, origin="lower", aspect="auto",
                    extent=[kfwd_g.min(), kfwd_g.max(), T_days_g.min(), T_days_g.max()],
                    cmap="viridis", vmin=0.05, vmax=0.70)
    ax0.scatter(kfwd, T_days_q, c=obs, cmap="plasma", s=12,
                edgecolors="white", linewidths=0.3,
                vmin=iv_g.min(), vmax=iv_g.max())
    ax0.axvline(1.0, color="white", linestyle="--", linewidth=0.8)
    plt.colorbar(im, ax=ax0, label="IV")
    ax0.set_xlabel("K/Fwd")
    ax0.set_ylabel("T (days)")
    ax0.set_title("IV Surface + Quotes")

    # 2. 3D surface
    ax1 = fig.add_subplot(gs[0, 1], projection="3d")
    ax1.plot_surface(KK, TT, iv_g, cmap="viridis", linewidth=0, alpha=0.85)
    ax1.scatter(kfwd, T_days_q, obs, color="black", s=6, depthshade=False)
    ax1.set_xlabel("K/Fwd", labelpad=3)
    ax1.set_ylabel("T (days)", labelpad=3)
    ax1.set_zlabel("IV", labelpad=3)
    ax1.set_title("3D Surface")

    # 3. Residual scatter
    ax2 = fig.add_subplot(gs[0, 2])
    vmax = max(np.abs(res).max(), 1e-4)
    sc = ax2.scatter(kfwd, T_days_q, c=res, cmap="RdBu_r",
                     s=15, vmin=-vmax, vmax=vmax)
    plt.colorbar(sc, ax=ax2, label="Pred-Obs IV")
    ax2.axvline(1.0, color="black", linestyle="--", linewidth=0.8)
    ax2.set_xlabel("K/Fwd")
    ax2.set_ylabel("T (days)")
    ax2.set_title(f"Residuals  RMSE={result.rmse:.4f}")

    # 4. Smile cross-sections at selected maturities
    ax3 = fig.add_subplot(gs[1, 0])
    target_mats = [30, 60, 90, 180]
    cmap_smile = plt.cm.plasma
    for j, tm in enumerate(target_mats):
        ti = int(np.argmin(np.abs(T_days_g - tm)))
        color = cmap_smile(j / len(target_mats))
        ax3.plot(kfwd_g, iv_g[ti, :], color=color, label=f"T≈{int(T_days_g[ti])}d", linewidth=1.5)
        # overlay quotes at similar maturity
        mask_q = np.abs(T_days_q - T_days_g[ti]) < 7
        if mask_q.sum() > 0:
            ax3.scatter(kfwd[mask_q], obs[mask_q], color=color, s=20, zorder=5)
    ax3.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
    ax3.set_xlabel("K/Fwd")
    ax3.set_ylabel("Implied Vol")
    ax3.set_title("Smile Cross-Sections")
    ax3.legend(fontsize=8)

    # 5. Term structure at ATM
    ax4 = fig.add_subplot(gs[1, 1])
    atm_vol = iv_g[:, atm_idx]
    ax4.plot(T_days_g, atm_vol, color="steelblue", linewidth=2, label="ATM IV (pred)")
    atm_mask = np.abs(kfwd - 1.0) < 0.05
    if atm_mask.sum() > 0:
        ax4.scatter(T_days_q[atm_mask], obs[atm_mask], color="black", s=20, zorder=5, label="ATM quotes")
    ax4.set_xlabel("T (days)")
    ax4.set_ylabel("ATM Implied Vol")
    ax4.set_title("ATM Term Structure")
    ax4.legend()

    # 6. Residual histogram
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.hist(res, bins=30, color="steelblue", edgecolor="white", linewidth=0.3)
    ax5.axvline(0, color="red", linestyle="--", linewidth=1.5)
    ax5.set_xlabel("Residual (pred-obs)")
    ax5.set_ylabel("Count")
    ax5.set_title(f"Residual Hist  bias={result.bias:.4f}")

    fig.suptitle(
        f"{title_prefix}Date: {result.date}  |  RMSE={result.rmse:.4f}  MAE={result.mae:.4f}  "
        f"R²={result.r2:.4f}  N={result.n_quotes}",
        fontsize=12, fontweight="bold"
    )
    plt.show()


rmse_vals = [r.rmse for r in results]
sorted_idx = np.argsort(rmse_vals)
best_idx   = sorted_idx[0]
median_idx = sorted_idx[len(sorted_idx)//2]
worst_idx  = sorted_idx[-1]

for idx, label in [(best_idx, "BEST — "), (median_idx, "MEDIAN — "), (worst_idx, "WORST — ")]:
    visualize_single_day(results[idx], label)

## 14 — Time-Series Stability

In [ ]:
dates_ts = pd.to_datetime(summary["date"])

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

def ts_plot(ax, dates, vals, title, ylabel, color="steelblue", rolling=5):
    ax.plot(dates, vals, alpha=0.5, color=color, linewidth=0.8, label="Daily")
    roll = pd.Series(vals, index=dates).rolling(rolling, center=True).mean()
    ax.plot(dates, roll, color="darkred", linewidth=1.5, label=f"{rolling}d MA")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)

ts_plot(axes[0,0], dates_ts, summary["rmse"],          "RMSE Over Time",         "RMSE")
ts_plot(axes[0,1], dates_ts, summary["bias"],          "Bias Over Time",         "Bias", "darkorchid")
ts_plot(axes[1,0], dates_ts, summary["r2"],            "R² Over Time",           "R²", "goldenrod")
ts_plot(axes[1,1], dates_ts, summary["pct_within_spread"],"Bid-Ask Hit Rate", "%", "seagreen")
ts_plot(axes[2,0], dates_ts, summary["arb_mono"] + summary["arb_convex"] + summary["arb_cal"],
        "Total No-Arb Penalty", "Penalty", "tomato")
ts_plot(axes[2,1], dates_ts, summary["n_quotes"], "Quote Count per Day", "N quotes", "slategray")

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.set_major_locator(plt.MaxNLocator(6))
        ax.tick_params(axis="x", rotation=20)

fig.suptitle("Validation Metrics Time-Series", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 15 — Residuals by IV Level

In [ ]:
iv_bins   = [0.00, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.60, 1.0]
iv_labels = ["0-10%", "10-15%", "15-20%", "20-25%", "25-30%", "30-40%", "40-60%", ">60%"]
iv_idx    = np.digitize(all_obs, iv_bins) - 1

iv_bucket_df = bucket_stats(iv_idx, iv_labels)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title, color in [
    (axes[0], "rmse", "RMSE by IV Level",   "steelblue"),
    (axes[1], "bias", "Bias by IV Level",   "darkorchid"),
    (axes[2], "p95",  "95th |Res| by IV",   "tomato"),
]:
    bars = ax.bar(iv_bucket_df["bucket"], iv_bucket_df[col], color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_xticklabels(iv_bucket_df["bucket"], rotation=30, ha="right")
    ax.set_xlabel("Observed IV")

fig.suptitle("Error Profile by IV Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print(iv_bucket_df.to_string(index=False, float_format="{:.5f}".format))

## 16 — Smoothness & Curvature Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(summary["curvature"], bins=40, color="teal", edgecolor="white")
ax.axvline(summary["curvature"].median(), color="red", linestyle="--",
           label=f"Median={summary['curvature'].median():.2e}")
ax.set_title("Surface Curvature Distribution")
ax.set_xlabel("L2 second-difference curvature")
ax.legend()

ax = axes[1]
ax.scatter(summary["curvature"], summary["rmse"], s=15, alpha=0.5, color="steelblue")
ax.set_xlabel("Curvature")
ax.set_ylabel("RMSE")
ax.set_title("Curvature vs RMSE")

plt.tight_layout()
plt.show()

# Show mean curvature map across all days
curv_u_sum = np.zeros((spec.Nv, spec.Nu - 2))
curv_v_sum = np.zeros((spec.Nv - 2, spec.Nu))
for r in results:
    iv_g = r.iv_grid
    curv_u_sum += (iv_g[:, 2:] - 2*iv_g[:, 1:-1] + iv_g[:, :-2])**2
    curv_v_sum += (iv_g[2:, :] - 2*iv_g[1:-1, :] + iv_g[:-2, :])**2

curv_u_mean = curv_u_sum / len(results)
curv_v_mean = curv_v_sum / len(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
u_arr_g = np.linspace(spec.u_min, spec.u_max, spec.Nu)
v_arr_g = np.linspace(spec.v_min, spec.v_max, spec.Nv)
kfwd_g  = np.exp(u_arr_g)
T_days_g = np.exp(v_arr_g) * 365.0

im0 = axes[0].imshow(curv_u_mean, origin="lower", aspect="auto",
                     extent=[kfwd_g[1], kfwd_g[-2], T_days_g.min(), T_days_g.max()],
                     cmap="hot")
plt.colorbar(im0, ax=axes[0], label="d²IV/du²")
axes[0].set_title("Mean Curvature in Moneyness Direction")
axes[0].set_xlabel("K/Fwd")
axes[0].set_ylabel("T (days)")

im1 = axes[1].imshow(curv_v_mean, origin="lower", aspect="auto",
                     extent=[kfwd_g.min(), kfwd_g.max(), T_days_g[1], T_days_g[-2]],
                     cmap="hot")
plt.colorbar(im1, ax=axes[1], label="d²IV/dv²")
axes[1].set_title("Mean Curvature in Maturity Direction")
axes[1].set_xlabel("K/Fwd")
axes[1].set_ylabel("T (days)")

plt.tight_layout()
plt.show()

## 17 — Attention Map Analysis

In [ ]:
from visualsurface.viz import extract_encoder_attention, plot_encoder_attention

sample_row = dm.val_ds.rows[0]
batch_sample = dm.collate_fn([sample_row]).to(DEVICE)

with torch.no_grad():
    attn = extract_encoder_attention(batch_sample.img, lit.model.img_enc)  # [1, H, Np, Np]

n_heads = attn.shape[1]
patch   = lit.hparams.get("patch", 4)
Nv_p = spec.Nv // patch
Nu_p = spec.Nu // patch

mean_recv = attn[0].mean(dim=1).numpy()       # [n_heads, Np]
maps = mean_recv.reshape(n_heads, Nv_p, Nu_p)  # [n_heads, Nv/p, Nu/p]

ncols = min(4, n_heads)
nrows = math.ceil(n_heads / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), squeeze=False)

u_arr_g = np.linspace(spec.u_min, spec.u_max, Nu_p)
v_arr_g = np.linspace(spec.v_min, spec.v_max, Nv_p)
kfwd_p  = np.exp(u_arr_g)
T_p     = np.exp(v_arr_g) * 365.0

for h in range(n_heads):
    row, col = h // ncols, h % ncols
    ax = axes[row][col]
    im = ax.imshow(maps[h], origin="lower", cmap="hot", aspect="auto",
                   extent=[kfwd_p.min(), kfwd_p.max(), T_p.min(), T_p.max()])
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f"Head {h}")
    ax.set_xlabel("K/Fwd")
    ax.set_ylabel("T (d)")

for h in range(n_heads, nrows*ncols):
    axes[h//ncols][h%ncols].axis("off")

fig.suptitle("ViT Encoder — Last Layer Mean Received Attention", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 18 — Model Parameter Analysis

In [ ]:
param_counts = {}
for name, module in lit.model.named_children():
    count = sum(p.numel() for p in module.parameters())
    param_counts[name] = count

fig, ax = plt.subplots(figsize=(10, 5))
names = list(param_counts.keys())
counts = [param_counts[n] for n in names]
colors_p = plt.cm.tab10(np.linspace(0, 1, len(names)))
bars = ax.barh(names, counts, color=colors_p)
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
            f"{count/1e6:.1f}M", va="center", fontsize=9)
ax.set_xlabel("Parameter Count")
ax.set_title(f"Parameter Distribution by Component\nTotal: {sum(counts)/1e6:.1f}M")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
plt.tight_layout()
plt.show()

print("\nComponent breakdown:")
for n, c in sorted(param_counts.items(), key=lambda x: -x[1]):
    print(f"  {n:<30} {c:>12,}  ({100*c/sum(counts):.1f}%)")
print(f"  {'TOTAL':<30} {sum(counts):>12,}")

## 19 — Trading Desk Readiness Scorecard

In [ ]:
# Thresholds — adjust to your desk's requirements
THRESHOLDS = {
    "rmse_median":        {"target": 0.010, "warn": 0.015, "unit": "IV",   "direction": "lower"},
    "rmse_p95":           {"target": 0.025, "warn": 0.040, "unit": "IV",   "direction": "lower"},
    "bias_abs_median":    {"target": 0.003, "warn": 0.008, "unit": "IV",   "direction": "lower"},
    "r2_median":          {"target": 0.990, "warn": 0.975, "unit": "",     "direction": "higher"},
    "bid_ask_pct_median": {"target": 0.600, "warn": 0.400, "unit": "%",    "direction": "higher"},
    "arb_total_median":   {"target": 1e-5,  "warn": 1e-4,  "unit": "",     "direction": "lower"},
    "curvature_median":   {"target": 5e-5,  "warn": 2e-4,  "unit": "",     "direction": "lower"},
}

arb_total = summary["arb_mono"] + summary["arb_convex"] + summary["arb_cal"]

measured = {
    "rmse_median":        float(summary["rmse"].median()),
    "rmse_p95":           float(summary["rmse"].quantile(0.95)),
    "bias_abs_median":    float(summary["bias"].abs().median()),
    "r2_median":          float(summary["r2"].median()),
    "bid_ask_pct_median": float(summary["pct_within_spread"].median()),
    "arb_total_median":   float(arb_total.median()),
    "curvature_median":   float(summary["curvature"].median()),
}

def score(key, val, thr):
    t, w = thr["target"], thr["warn"]
    if thr["direction"] == "lower":
        if val <= t:   return "PASS",   "#2ecc71"
        elif val <= w: return "WARN",   "#f39c12"
        else:          return "FAIL",   "#e74c3c"
    else:
        if val >= t:   return "PASS",   "#2ecc71"
        elif val >= w: return "WARN",   "#f39c12"
        else:          return "FAIL",   "#e74c3c"

rows = []
for key, val in measured.items():
    thr = THRESHOLDS[key]
    status, color = score(key, val, thr)
    rows.append({
        "Metric":    key.replace("_", " ").title(),
        "Value":     f"{val:.5g}",
        "Target":    f"{thr['target']:.5g}",
        "Warn":      f"{thr['warn']:.5g}",
        "Direction": thr["direction"],
        "Status":    status,
        "_color":    color,
    })

scorecard = pd.DataFrame(rows)

# Pretty print table
print("\n" + "="*70)
print("   TRADING DESK READINESS SCORECARD")
print("="*70)
header = f"{'Metric':<28} {'Value':>10} {'Target':>10} {'Warn':>10} {'Status':>8}"
print(header)
print("-"*70)
pass_count, warn_count, fail_count = 0, 0, 0
for _, row in scorecard.iterrows():
    line = f"{row['Metric']:<28} {row['Value']:>10} {row['Target']:>10} {row['Warn']:>10} {row['Status']:>8}"
    print(line)
    if row["Status"] == "PASS": pass_count += 1
    elif row["Status"] == "WARN": warn_count += 1
    else: fail_count += 1
print("-"*70)
print(f"  PASS: {pass_count}  WARN: {warn_count}  FAIL: {fail_count}")
print("="*70)

# Visual scorecard
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
col_labels = ["Metric", "Measured", "Target", "Warn Level", "Status"]
table_data = [
    [r["Metric"], r["Value"], r["Target"], r["Warn"], r["Status"]]
    for _, r in scorecard.iterrows()
]
tbl = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc="center",
    cellLoc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)

# Color status cells
for i, row in enumerate(rows):
    cell = tbl[i+1, 4]
    cell.set_facecolor(row["_color"])
    cell.set_text_props(color="white", fontweight="bold")
# Header style
for j in range(5):
    tbl[0, j].set_facecolor("#2c3e50")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

fig.suptitle("Trading Desk Readiness Scorecard", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 20 — Complete Metrics Summary Export

In [ ]:
export_path = "evaluation_results.csv"
summary.to_csv(export_path, index=False)
print(f"Per-day results saved to: {export_path}")

print("\n=== FULL METRIC SUMMARY ===")
print(f"Checkpoint      : {CHECKPOINT_PATH}")
print(f"Data            : {DATA_PATH}")
print(f"Val days        : {len(results)}")
print(f"Total quotes    : {len(all_res):,}")
print()
print("── Fitting ──────────────────────────────")
print(f"  RMSE (median/mean/p95) : {summary['rmse'].median():.5f} / {summary['rmse'].mean():.5f} / {summary['rmse'].quantile(0.95):.5f}")
print(f"  MAE  (median/mean)     : {summary['mae'].median():.5f} / {summary['mae'].mean():.5f}")
print(f"  Bias (median)          : {summary['bias'].median():.5f}")
print(f"  R²   (median)          : {summary['r2'].median():.6f}")
print()
print("── Calibration ──────────────────────────")
print(f"  Within bid-ask (median): {100*summary['pct_within_spread'].median():.2f}%")
print()
print("── No-Arbitrage ─────────────────────────")
print(f"  Monotonicity  (median) : {summary['arb_mono'].median():.2e}")
print(f"  Convexity     (median) : {summary['arb_convex'].median():.2e}")
print(f"  Calendar      (median) : {summary['arb_cal'].median():.2e}")
print(f"  Total         (median) : {arb_total.median():.2e}")
print()
print("── Smoothness ───────────────────────────")
print(f"  Curvature     (median) : {summary['curvature'].median():.2e}")
print()
print("── Training ─────────────────────────────")
print(f"  Parameters             : {sum(p.numel() for p in lit.parameters()):,}")
print(f"  Hidden size            : {lit.hparams.get('hidden_size', lit.hparams.get('d_model', '?'))}")
print(f"  ViT layers/heads       : {lit.hparams.get('vit_layers','?')}/{lit.hparams.get('vit_heads','?')}")
print(f"  Decoder layers/heads   : {lit.hparams.get('dec_layers','?')}/{lit.hparams.get('dec_heads','?')}")